In [1]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# Monitoring training

Real datasets can span millions of points across thousands of features. Fitting glass box UMAP at that scale takes real time, and in such cases you'll want to observe how training is proceeding. This guide shows you how to monitor training progress and keep a record of each fit on disk.

## Automated logging with Tensorboard

Under the hood, glass box UMAP trains its encoder with [PyTorch Lightning](https://lightning.ai/), and every fit is automatically logged with [TensorBoard](https://www.tensorflow.org/tensorboard) to a temporary directory. To persist these logs, just pass an explicit `checkpoint_dir` to {class}`GlassBoxUMAP <glass_box_umap.GlassBoxUMAP>`:

In [2]:
from pathlib import Path
import shutil

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from glass_box_umap import GlassBoxUMAP

# Store logs to ./runs/
checkpoint_dir = Path.cwd() / "runs"

# The directory can exist, but needn't. We remove it for a fresh start.
shutil.rmtree(checkpoint_dir)

embedder = GlassBoxUMAP(
    random_state=0,
    checkpoint_dir=checkpoint_dir,
    quiet=True,
)

X, y = load_digits(return_X_y=True)
X = StandardScaler().fit_transform(X)
embedder.fit(X)

!tree runs/

runs/
├── checkpoints
│   └── best.ckpt
└── logs
    ├── events.out.tfevents.1778027879.evans-Apple-MacBook-Pro.28269.0
    └── hparams.yaml

3 directories, 3 files


- `checkpoints/best.ckpt` is the same checkpoint that `restore_best_weights` (default `True`) reloads at the end of training, so you don't normally need to touch it. It stays on disk in case you want to inspect or reload a specific run later.
- `logs/events.out.tfevents.…` is the TensorBoard event file.

The TensorBoard events file isn't human-readable, but can be viewed using a tensoboard server:

```bash
tensorboard --logdir runs/
```

:::{note}
`tensorboard` ships as a dependency of `glass-box-umap`, so nothing extra needs to be installed. From the project root:
:::

That starts a server (default `http://localhost:6006`) which auto-discovers every event file under `runs/`. Leave it running while you train and it will poll the directory and refresh logged data as new events are written, allowing you to watch a fit in progress.

## Visualizing embedding evolution during training

As an alternative diagnostic, glass box UMAP exposes a {class}`LiveEmbeddingCallback <glass_box_umap.plotting.LiveEmbeddingCallback>` that streams the embedding itself to a Bokeh server in your browser. After each training epoch, it runs `transform` on a slice of `X` and pushes the new 2D coordinates to the page, where a slider lets you scrub back through every epoch, a play button replays the trajectory, and a save button writes a self-contained HTML snapshot of the run.

:::{warning}
Running `transform` after every epoch is not free. On large datasets the extra forward pass per epoch will noticeably slow training, so pass a representative subsample to the callback (as above) rather than the full `X`. When all you need is the loss curve, prefer TensorBoard.
:::

This live embedding offers diagnostic insight into how the learned manifold is forming. You can watch the geometry settle (or fail to) and catch a misbehaving run within a few epochs instead of waiting for the training to complete.

It plugs in through `extra_callbacks`:

:::{admonition} Install the plotting extras
:class: tip dropdown

`glass_box_umap.plotting` is an optional dependency that's required for this feature. It can be installed like so:

```bash
pip install "glass-box-umap[plotting]"
# or
uv pip install "glass-box-umap[plotting]"
```
:::

:::{admonition} LiveEmbeddingCallback API
:class: api, dropdown

From the {class}`API docs <glass_box_umap.plotting.LiveEmbeddingCallback>`:

```{eval-rst}
.. automethod:: glass_box_umap.plotting.LiveEmbeddingCallback
    :noindex:
```
:::

In [3]:
from glass_box_umap.plotting import LiveEmbeddingCallback

# Create the embedder
embedder = GlassBoxUMAP(
    random_state=0,
    quiet=True
)

# Now create the callback, passing embedder.transform
labels = [str(idx) for idx in y]
callback = LiveEmbeddingCallback(
    transform_fn=embedder.transform,
    X=X[:500],
    labels=labels[:500],
    block_after_fit=False,
)

# Append the callback to `extra_callbacks`
embedder.extra_callbacks.append(callback)

_ = embedder.fit(X)

Live embedding serving at http://localhost:52731/
Training done. Server still serving at http://localhost:52731/.


:::{note}
The above code will open an interface in your browser, updating after each epoch. For your convenience, we replicate the interface below.
:::

In [4]:
from html import escape
from pathlib import Path

from IPython.display import HTML

artifact = Path("../_static/monitoring_training_example.html")

embedded = HTML(
    f'<iframe srcdoc="{escape(artifact.read_text(), quote=True)}" '
    f'width="100%" height="750" style="border:none"></iframe>'
)
embedded

Hit "Play" to observe the embedding evolve throughout the training.